# RT Reviewer Fixes Runner  
**Optimized for L40S × 7 on RunPod — maximum speed**

| # | Workload | Closes reviewer objection |
|---|---|---|
| A | LongLLMLingua baseline — hard stress set | "no strong NLP baseline" |
| B | Full LongMemEval-S (no 40-turn cap) | "truncated conversations mislead" |
| C | LongLLMLingua baseline — MSC valid | cross-benchmark baseline coverage |
| D | Llama-3.2-3B scale validation | "only one 3B model tested" |

**Run order:** cells 0→12 top-to-bottom once.  
**Resumable:** if a workload cell crashes, just re-run it — completed shards are auto-skipped.  
**Monitoring:** run the `monitor_progress()` cell or `sanity_check()` cell at any time mid-run.

## 0 · Configuration — edit this cell first

In [ ]:
from pathlib import Path
import os

# ── repo ─────────────────────────────────────────────────────────────────────
REPO_URL   = "https://github.com/SteveMama/rt-geometry-memory.git"
REPO_DIR   = Path("/workspace/RT").resolve()

# ── credentials ──────────────────────────────────────────────────────────────
GITHUB_USER  = ""   # your GitHub username
GITHUB_TOKEN = ""   # PAT with repo write scope
HF_TOKEN     = ""   # HuggingFace token — required only for Llama-3.2-3B

# ── experiment config ─────────────────────────────────────────────────────────
MODEL_KEY          = "qwen25_15b"
BUDGETS            = "0.20,0.35,0.50"
GPU_COUNT          = 0       # 0 = all visible GPUs
JOB_MULTIPLIER     = 3       # shards per GPU — 3×7 = 21 shards → better load balance
EXTRACT_BATCH_SIZE = 64      # L40S 48GB VRAM handles 64 easily; drop to 32 for A100 40GB
RUN_PREFIX         = "reviewer_fixes"

# ── workload toggles ──────────────────────────────────────────────────────────
RUN_HARDSET_BASELINES = True
RUN_FULL_LME          = True
RUN_MSC_BASELINES     = True
RUN_LLAMA32_3B        = True

# ── push ─────────────────────────────────────────────────────────────────────
AUTO_PUSH = True

print(f"REPO_DIR  : {REPO_DIR}")
print(f"Model     : {MODEL_KEY}  Budgets: {BUDGETS}")
print(f"Batch size: {EXTRACT_BATCH_SIZE}  GPU: {GPU_COUNT or 'all'}  Shards/GPU: {JOB_MULTIPLIER}")
print(f"Workloads : hardset={RUN_HARDSET_BASELINES}  fullLME={RUN_FULL_LME}  "
      f"msc={RUN_MSC_BASELINES}  llama32={RUN_LLAMA32_3B}")

## 1 · Helpers

In [ ]:
import subprocess, sys, time, json, threading

def run(cmd, cwd=None, env=None, check=True):
    print(f"\n$ {cmd}", flush=True)
    proc = subprocess.Popen(
        cmd, shell=True, cwd=str(cwd) if cwd else None, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed ({proc.returncode}): {cmd}")
    return proc.returncode

def env_with(**extras):
    merged = {**os.environ}
    for k, v in extras.items():
        if v: merged[k] = str(v)
    return merged

def workload_env(overrides=None):
    """Base env for all workload cells."""
    e = env_with(
        PYTHON_BIN=str(PY),
        MODEL_KEY=MODEL_KEY,
        BUDGETS=BUDGETS,
        GPU_COUNT=str(n_gpus),
        JOB_MULTIPLIER=str(JOB_MULTIPLIER),
        EXTRACT_BATCH_SIZE=str(EXTRACT_BATCH_SIZE),
        RUN_PREFIX=RUN_PREFIX,
        HF_TOKEN=HF_TOKEN,
        HUGGING_FACE_HUB_TOKEN=HF_TOKEN,
        TOKENIZERS_PARALLELISM="false",
        PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
        GITHUB_USER=GITHUB_USER,
        GITHUB_TOKEN=GITHUB_TOKEN,
        SKIP_DOWNLOAD="1",
        SKIP_PUSH="1",
        HARDSET_INPUT=str(HARDSET),
        MSC_INPUT=str(MSC_JSONL) if 'MSC_JSONL' in dir() else "",
        LONGMEM_INPUT=str(LME_JSONL) if 'LME_JSONL' in dir() else "",
    )
    if overrides:
        e.update(overrides)
    return e

print("helpers ready")

## 2 · Clone / pull repo

In [ ]:
if not REPO_DIR.exists():
    clone_url = REPO_URL
    if GITHUB_USER and GITHUB_TOKEN:
        proto, rest = REPO_URL.split("://", 1)
        clone_url = f"{proto}://{GITHUB_USER}:{GITHUB_TOKEN}@{rest}"
    run(f"git clone {clone_url} {REPO_DIR}")
else:
    print(f"{REPO_DIR} exists — pulling latest")
    run("git pull --ff-only", cwd=REPO_DIR, check=False)

run("git log --oneline -5", cwd=REPO_DIR)

## 3 · Install dependencies

In [ ]:
VENV = REPO_DIR / ".venv"
PY   = VENV / "bin" / "python"

if not PY.exists():
    run(f"python3 -m venv {VENV}")
run(f"{PY} -m pip install -q --upgrade pip", cwd=REPO_DIR)
run(f"PYTHON_BIN={PY} bash scripts/install_reviewer_deps.sh", cwd=REPO_DIR)

# matplotlib for plots
run(f"{PY} -m pip install -q matplotlib pandas scipy", cwd=REPO_DIR)

## 4 · Verify GPU + torch + flash-attn

In [ ]:
run("nvidia-smi")
run(f"""{PY} -c """
    "\"import torch; "
    "print(f'torch={torch.__version__}'); "
    "print(f'cuda={torch.cuda.is_available()}'); "
    "[print(f'  [{i}] {torch.cuda.get_device_name(i)} "
     "({torch.cuda.get_device_properties(i).total_memory//1024**3} GB)') "
     "for i in range(torch.cuda.device_count())]\""
)
run(f"{PY} -c \"import flash_attn; print('flash-attn', flash_attn.__version__)\"", check=False)
run(f"{PY} -c \"from llmlingua import PromptCompressor; print('llmlingua ok')\"")

## 5 · Detect GPUs

In [ ]:
result = subprocess.run(
    "nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader",
    shell=True, capture_output=True, text=True
)
gpu_rows = [r.strip() for r in result.stdout.strip().splitlines() if r.strip()]
print("Available GPUs:")
for r in gpu_rows: print(f"  {r}")

n_gpus = GPU_COUNT if GPU_COUNT > 0 else len(gpu_rows)
JOB_SHARDS = n_gpus * JOB_MULTIPLIER
print(f"\nUsing {n_gpus} GPU(s)  ×  {JOB_MULTIPLIER} shards = {JOB_SHARDS} total shards/workload")
print(f"EXTRACT_BATCH_SIZE = {EXTRACT_BATCH_SIZE}")

## 6 · HuggingFace login (Llama-3.2-3B only)

In [ ]:
if HF_TOKEN:
    run(f"{PY} -c \"from huggingface_hub import login; login('{HF_TOKEN}')\"")
    print("HuggingFace login ok")
else:
    print("HF_TOKEN not set — Llama-3.2-3B workload will be skipped")

## 7 · Download benchmarks

In [ ]:
BENCH     = REPO_DIR / "benchmarks"
BENCH.mkdir(parents=True, exist_ok=True)
MSC_JSONL = BENCH / "msc_valid_normalized.jsonl"

if not MSC_JSONL.exists():
    run(f"{PY} scripts/download_public_benchmark.py "
        f"--benchmark msc_valid --output {BENCH}/msc_valid_raw.jsonl", cwd=REPO_DIR)
    run(f"{PY} scripts/prepare_public_benchmark_jsonl.py "
        f"--format msc --input {BENCH}/msc_valid_raw.jsonl "
        f"--output {MSC_JSONL} --family msc_valid", cwd=REPO_DIR)

n_msc = sum(1 for _ in open(MSC_JSONL))
print(f"MSC valid: {n_msc} conversations")

In [ ]:
# Full LongMemEval-S — NO --max-turns-per-conversation (fixes the truncation objection)
LME_JSONL = BENCH / "longmemeval_s_full_normalized.jsonl"

if not LME_JSONL.exists():
    run(f"{PY} scripts/download_public_benchmark.py "
        f"--benchmark longmemeval_s_cleaned --output {BENCH}/longmemeval_s_raw.json",
        cwd=REPO_DIR)
    run(f"{PY} scripts/prepare_public_benchmark_jsonl.py "
        f"--format longmemeval --input {BENCH}/longmemeval_s_raw.json "
        f"--output {LME_JSONL} --family longmemeval_s_full", cwd=REPO_DIR)

n_lme = sum(1 for _ in open(LME_JSONL))
print(f"LongMemEval-S full: {n_lme} conversations (no turn cap)")

In [ ]:
HARDSET = REPO_DIR / "paper1_geometry" / "assets" / "paper2_behavior_stress_conversations.jsonl"
assert HARDSET.exists(), f"hard stress set missing: {HARDSET}"
n_hs = sum(1 for _ in open(HARDSET))
print(f"Hard stress set: {n_hs} conversations")

---
## 8 · Live Monitor + Sanity Check
**Run these cells at any time while workloads are running.**

In [ ]:
def monitor_progress(results_root=None):
    """Print a live snapshot of shard progress and partial scores."""
    import pandas as pd
    root = Path(results_root or REPO_DIR / "results" / "reviewer_fixes")

    # shard progress
    progress_files = sorted(root.rglob("progress.json"))
    done = 0; total = len(progress_files)
    for pf in progress_files:
        try:
            p = json.loads(pf.read_text())
            if p.get("status") == "complete": done += 1
        except Exception: pass

    print(f"\n{'─'*60}")
    print(f"  Shards  : {done}/{total} complete")

    # partial evaluation rows
    frames = []
    for csv in sorted(root.rglob("evaluation_rows.csv")):
        try:
            df = pd.read_csv(csv)
            df["_tag"] = csv.relative_to(root).parts[0]
            frames.append(df)
        except Exception: pass

    if not frames:
        print("  No evaluation rows yet")
        print(f"{'─'*60}\n")
        return

    df = pd.concat(frames, ignore_index=True)
    print(f"  Eval rows: {len(df):,}")

    if "top1_match" in df.columns:
        agg = (df.groupby(["_tag", "policy_name", "budget_fraction"])["top1_match"]
               .agg(["mean", "count"]).reset_index())
        agg.columns = ["benchmark", "policy", "budget", "mean_score", "n"]
        agg["mean_score"] = agg["mean_score"].round(3)
        print(f"\n  Partial scores (top1_match):")
        print(agg.to_string(index=False))

    if "token_fraction" in df.columns:
        budget_err = (df["token_fraction"] - df["budget_fraction"]).abs().mean()
        print(f"\n  Budget adherence (mean |err|): {budget_err:.3f}")

    print(f"{'─'*60}\n")


# run immediately
monitor_progress()

In [ ]:
# Sanity check — run mid-experiment to catch degenerate geometry / codec failures
rc = run(
    f"{PY} scripts/sanity_check.py "
    f"--results-root {REPO_DIR}/results/reviewer_fixes "
    f"--expected-policies uniform,longllmlingua,geometry_keep_compress_drop,semantic_keep_compress_drop",
    cwd=REPO_DIR, check=False
)
if rc == 2:
    print("\n⚠️  CRITICAL issues found — inspect logs before continuing")
elif rc == 1:
    print("\n⚠️  Warnings (experiment can continue — recheck after more shards complete)")
else:
    print("\n✅  All sanity checks passed")

---
## Workload A · LongLLMLingua baseline — hard stress set  
*Closes: "no strong NLP compression baseline"*

In [ ]:
if RUN_HARDSET_BASELINES:
    PLAN_A = REPO_DIR / "results" / "reviewer_fixes" / "shard_plans" / "baselines_hardset"
    PLAN_A.mkdir(parents=True, exist_ok=True)
    run(f"{PY} -m paper3_codec.plan_conversation_shards "
        f"--input-path {HARDSET} --shard-count {JOB_SHARDS} "
        f"--target-turn-stride 1 --output-dir {PLAN_A}", cwd=REPO_DIR)
    print(f"Planned {JOB_SHARDS} shards → {PLAN_A}")

In [ ]:
if RUN_HARDSET_BASELINES:
    run(
        f"bash scripts/run_reviewer_fixes_multigpu.sh",
        cwd=REPO_DIR,
        env=env_with(
            **workload_env(),
            RUN_HARDSET_BASELINES="1", RUN_FULL_LME="0",
            RUN_MSC_BASELINES="0",    RUN_LLAMA32_3B="0",
            HARDSET_INPUT=str(HARDSET),
            MSC_INPUT=str(MSC_JSONL),
            LONGMEM_INPUT=str(LME_JSONL),
        )
    )
    print("Workload A complete")

In [ ]:
# Quick sanity after Workload A
if RUN_HARDSET_BASELINES:
    monitor_progress()
    run(f"{PY} scripts/sanity_check.py "
        f"--results-root {REPO_DIR}/results/reviewer_fixes/baselines",
        cwd=REPO_DIR, check=False)

---
## Workload B · Full LongMemEval-S (no 40-turn cap)  
*Closes: "40-turn truncation of 400–600 turn conversations is misleading"*

In [ ]:
if RUN_FULL_LME:
    PLAN_B = REPO_DIR / "results" / "reviewer_fixes" / "shard_plans" / "fullLME"
    PLAN_B.mkdir(parents=True, exist_ok=True)
    run(f"{PY} -m paper3_codec.plan_conversation_shards "
        f"--input-path {LME_JSONL} --shard-count {JOB_SHARDS} "
        f"--target-turn-stride 1 --output-dir {PLAN_B}", cwd=REPO_DIR)
    print(f"Planned {JOB_SHARDS} shards → {PLAN_B}")

In [ ]:
if RUN_FULL_LME:
    run(
        f"bash scripts/run_reviewer_fixes_multigpu.sh",
        cwd=REPO_DIR,
        env=env_with(
            **workload_env(),
            RUN_HARDSET_BASELINES="0", RUN_FULL_LME="1",
            RUN_MSC_BASELINES="0",    RUN_LLAMA32_3B="0",
            HARDSET_INPUT=str(HARDSET),
            MSC_INPUT=str(MSC_JSONL),
            LONGMEM_INPUT=str(LME_JSONL),
        )
    )
    print("Workload B complete")

In [ ]:
if RUN_FULL_LME:
    monitor_progress()
    run(f"{PY} scripts/sanity_check.py "
        f"--results-root {REPO_DIR}/results/reviewer_fixes/fullLME",
        cwd=REPO_DIR, check=False)

---
## Workload C · LongLLMLingua baseline — MSC valid

In [ ]:
if RUN_MSC_BASELINES:
    PLAN_C = REPO_DIR / "results" / "reviewer_fixes" / "shard_plans" / "baselines_msc"
    PLAN_C.mkdir(parents=True, exist_ok=True)
    run(f"{PY} -m paper3_codec.plan_conversation_shards "
        f"--input-path {MSC_JSONL} --shard-count {JOB_SHARDS} "
        f"--target-turn-stride 1 --output-dir {PLAN_C}", cwd=REPO_DIR)
    print(f"Planned {JOB_SHARDS} shards → {PLAN_C}")

In [ ]:
if RUN_MSC_BASELINES:
    run(
        f"bash scripts/run_reviewer_fixes_multigpu.sh",
        cwd=REPO_DIR,
        env=env_with(
            **workload_env(),
            RUN_HARDSET_BASELINES="0", RUN_FULL_LME="0",
            RUN_MSC_BASELINES="1",    RUN_LLAMA32_3B="0",
            HARDSET_INPUT=str(HARDSET),
            MSC_INPUT=str(MSC_JSONL),
            LONGMEM_INPUT=str(LME_JSONL),
        )
    )
    print("Workload C complete")

In [ ]:
if RUN_MSC_BASELINES:
    monitor_progress()
    run(f"{PY} scripts/sanity_check.py "
        f"--results-root {REPO_DIR}/results/reviewer_fixes/baselines",
        cwd=REPO_DIR, check=False)

---
## Workload D · Llama-3.2-3B scale validation  
*Closes: "only one 3B model tested"*

In [ ]:
if RUN_LLAMA32_3B:
    if not HF_TOKEN:
        print("WARNING: HF_TOKEN not set — skipping Llama-3.2-3B")
    else:
        run(
            f"bash scripts/run_reviewer_fixes_multigpu.sh",
            cwd=REPO_DIR,
            env=env_with(
                **workload_env(),
                RUN_HARDSET_BASELINES="0", RUN_FULL_LME="0",
                RUN_MSC_BASELINES="0",    RUN_LLAMA32_3B="1",
                HARDSET_INPUT=str(HARDSET),
                MSC_INPUT=str(MSC_JSONL),
                LONGMEM_INPUT=str(LME_JSONL),
            )
        )
        print("Workload D complete")

In [ ]:
if RUN_LLAMA32_3B and HF_TOKEN:
    monitor_progress()
    run(f"{PY} scripts/sanity_check.py "
        f"--results-root {REPO_DIR}/results/reviewer_fixes/scale_llama32_3b",
        cwd=REPO_DIR, check=False)

---
## 9 · Generate all plots

Produces 7 PNGs + a combined PDF in `results/reviewer_fixes/plots/`:

| File | Content |
|---|---|
| `01_score_curves.png` | top1_match vs budget, per policy, per benchmark |
| `02_score_boxplots.png` | score distribution by policy × budget |
| `03_kcd_action_breakdown.png` | keep / compress / evict fractions |
| `04_budget_adherence.png` | target vs achieved token fraction |
| `05_head_to_head.png` | Geometry KCD vs LongLLMLingua per-conversation |
| `06_geometry_signal.png` | geometry score distribution by codec action |
| `07_behavior_logprob.png` | answer neg-logprob delta by policy |
| `combined_report.pdf` | all panels in one file |

In [ ]:
PLOT_DIR = REPO_DIR / "results" / "reviewer_fixes" / "plots"
run(
    f"{PY} scripts/plot_reviewer_results.py "
    f"--results-root {REPO_DIR}/results/reviewer_fixes "
    f"--output-dir {PLOT_DIR}",
    cwd=REPO_DIR
)

# display inline
from IPython.display import Image, display
for png in sorted(PLOT_DIR.glob("0*.png")):
    print(f"\n── {png.name} ──")
    display(Image(filename=str(png)))

## 10 · Final sanity check (all workloads)

In [ ]:
rc = run(
    f"{PY} scripts/sanity_check.py "
    f"--results-root {REPO_DIR}/results/reviewer_fixes "
    f"--expected-policies uniform,longllmlingua,geometry_keep_compress_drop,"
    f"semantic_keep_compress_drop,semantic_query_conditioned_geometry_keep_compress_drop",
    cwd=REPO_DIR, check=False
)
print({0: "✅ All good", 1: "⚠️  Warnings only", 2: "🚨 Critical failures"}.get(rc, "?"))

## 11 · Commit & push to GitHub

In [ ]:
import datetime

if not AUTO_PUSH:
    print("AUTO_PUSH=False — skipping")
elif not GITHUB_USER or not GITHUB_TOKEN:
    print("Set GITHUB_USER and GITHUB_TOKEN in cell 0")
else:
    run(
        f"GITHUB_USER={GITHUB_USER} GITHUB_TOKEN={GITHUB_TOKEN} "
        f"bash scripts/colab_commit_push.sh "
        f"SteveMama pranav@vizit.com "
        f"'reviewer-fix results: longllmlingua baseline, full LME, llama32_3b "
        f"[{datetime.date.today()}]' "
        f"results/reviewer_fixes "
        f"notebooks/rt_reviewer_fixes_runner.ipynb",
        cwd=REPO_DIR
    )
    print("Pushed ✓")

## 12 · Package for download

In [ ]:
import shutil
archive = shutil.make_archive(
    str(REPO_DIR / f"reviewer_fixes_results_{RUN_PREFIX}"),
    "zip",
    root_dir=str(REPO_DIR / "results"),
    base_dir="reviewer_fixes",
)
size_mb = Path(archive).stat().st_size / 1e6
print(f"Archive: {archive}  ({size_mb:.1f} MB)")
print("Download from RunPod file manager or: scp root@<pod-ip>:{archive} ./")